Задание Pro: Реализуйте асинхронный Python-скрипт, который выполняет следующее:

1. Собирает списки популярных или бесплатных фильмов с двух видеосервисов, извлеките название фильма и ссылку на страницу (достаточно первые 10 фильмов):

Kion — страница: https://kion.ru

Wink — страница: https://wink.ru/collections

2. Собирает список научных статей по тематике "Компьютерные науки" с любого тематического сайта

Извлеките название статьи и ссылку на неё. (Можно ограничиться первыми 20 статьями.)

3. Реализует асинхронную архитектуру:
Все запросы и парсинг должны выполняться параллельно через asyncio.gather.

4. Сохраняет полученные данные в файл txt и выводит их в консоль.

In [2]:
#@title Установка зависимостей
!pip install -q aiohttp beautifulsoup4 fake-useragent lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 3.9 MB/s eta 0:00:00


In [26]:
#@title Kion
# Импорт необходимых библиотек
import nest_asyncio  # Для работы asyncio в Jupyter Notebook
nest_asyncio.apply()  # Применяем патч для обхода ограничений Jupyter

import time  # Для замера времени выполнения
from bs4 import BeautifulSoup  # Парсинг HTML/XML
from fake_useragent import UserAgent  # Генерация случайных User-Agent
from urllib.parse import urljoin  # Обработка относительных URL
from asyncio import gather
from aiohttp import ClientSession

async def fetch_url(url, session):
    """
    Асинхронно получает HTML-контент страницы
    Args:
        session (aiohttp.ClientSession): Асинхронная HTTP-сессия
        url (str): URL для запроса
    Returns:
        str: HTML-контент или None при ошибке
    """
    ua = UserAgent()  # Инициализация генератора User-Agent
    headers = {'User-Agent': ua.random}  # Случайный заголовок для каждого запроса

    try:
        # Асинхронный GET-запрос с таймаутом 10 секунд
        async with session.get(url, headers=headers, timeout=10) as response:
            if response.status == 200:  # Проверка успешного статуса
                return await response.text()  # Чтение контента с освобождением event loop
            else:
                # Логирование HTTP-ошибок
                print(f"Ошибка {response.status} при запросе к {url}")
                return None
    except Exception as e:
        # Перехват всех исключений (таймаут, сетевые ошибки и т.д.)
        print(f"Ошибка при запросе: {e}")
        return None

async def parse_kion(session):
    """
    Основная функция парсинга фильмов kion
    Returns:
        list: Список словарей с заголовками и ссылками
    """
    base_url = 'https://kion.ru/'  # Базовый URL сайта

    # Создание асинхронной HTTP-сессии
    html = await fetch_url(base_url, session)  # Асинхронный запрос
    if not html:
        return []  # Возвращаем пустой список при ошибке

    # Инициализация парсера с lxml (требует установки lxml)
    soup = BeautifulSoup(html, 'lxml')
    kion_list = []  # Контейнер для результатов

    # Поиск новостных блоков (основной и альтернативный селекторы)
    films_blocks = soup.find_all('a', class_='carousel__slide')

    # Обработка первых 10 элементов
    for block in films_blocks[:10]:
        try:
            # Поиск заголовка в разных вариантах верстки
            title = block.find('h3', class_='title')

            # Обработка ссылок для разных типов блоков
            link = block.get('href')
            if not link and block.name == 'div':  # Если блок - div
                parent_link = block.find_parent('a')  # Ищем родительскую ссылку
                link = parent_link.get('href') if parent_link else None

            # Конвертация относительных ссылок в абсолютные
            if link and not link.startswith(('http', '//')):
                link = urljoin(base_url, link)

            if title and link:
                kion_list.append({
                    'title': title.get_text(strip=True),  # Чистый текст
                    'url': link  # Полный URL
                })
        except Exception as e:
            print(f"Ошибка обработки блока: {str(e)}")
            continue  # Продолжаем при ошибках парсинга

    return kion_list

async def parse_winks(session):
    """
    Основная функция парсинга фильмов winks
    Returns:
        list: Список словарей с заголовками и ссылками
    """
    base_url = 'https://wink.ru/collections'  # Базовый URL сайта

    # Создание асинхронной HTTP-сессии
    html = await fetch_url(base_url, session)  # Асинхронный запрос
    if not html:
        return []  # Возвращаем пустой список при ошибке

    # Инициализация парсера с lxml (требует установки lxml)
    soup = BeautifulSoup(html, 'lxml')
    winks_list = []  # Контейнер для результатов

    # Поиск новостных блоков (основной и альтернативный селекторы)
    collect_blocks = soup.find_all('a', attrs={"data-type": "collection"})

    # Обработка первых 10 элементов
    for block in collect_blocks[:10]:
        try:
            # Поиск заголовка в разных вариантах верстки
            title = block.find('h4', attrs={"data-test": "cc-title"})

            # Обработка ссылок для разных типов блоков
            link = block.get('href')
            if not link and block.name == 'div':  # Если блок - div
                parent_link = block.find_parent('a')  # Ищем родительскую ссылку
                link = parent_link.get('href') if parent_link else None

            # Конвертация относительных ссылок в абсолютные
            if link and not link.startswith(('http', '//')):
                link = urljoin(base_url, link)

            if title and link:
                winks_list.append({
                    'title': title.get_text(strip=True),  # Чистый текст
                    'url': link  # Полный URL
                })
        except Exception as e:
            print(f"Ошибка обработки блока: {str(e)}")
            continue  # Продолжаем при ошибках парсинга

    return winks_list

async def computer_science(session):
    """
    Основная функция парсинга сайта компьютерных наук
    Returns:
        list: Список словарей с заголовками и ссылками
    """
    # base_url = 'https://www.computerra.ru'  # Базовый URL сайта
    base_url = 'https://www.computerra.ru/category/ai-bigdata/'  # Базовый URL сайта

    # Создание асинхронной HTTP-сессии
    html = await fetch_url(base_url, session)  # Асинхронный запрос
    if not html:
        return []  # Возвращаем пустой список при ошибке

    # Инициализация парсера с lxml (требует установки lxml)
    soup = BeautifulSoup(html, 'lxml')
    winks_list = []  # Контейнер для результатов

    # Поиск новостных блоков (основной и альтернативный селекторы)
    collect_blocks = soup.find_all("article", class_="home-category-item")

    # Обработка первых 20 элементов
    for block in collect_blocks[:20]:
        try:
            # Поиск заголовка в разных вариантах верстки
            a_tag = block.find("a")
            title = block.find("h3", class_="home-category-item__title")

            # Обработка ссылок для разных типов блоков
            link = a_tag.get('href')
            if not link and block.name == 'div':  # Если блок - div
                parent_link = block.find_parent('a')  # Ищем родительскую ссылку
                link = parent_link.get('href') if parent_link else None

            # Конвертация относительных ссылок в абсолютные
            if link and not link.startswith(('http', '//')):
                link = urljoin(base_url, link)

            if title and link:
                winks_list.append({
                    'title': title.get_text(strip=True),  # Чистый текст
                    'url': link  # Полный URL
                })
        except Exception as e:
            print(f"Ошибка обработки блока: {str(e)}")
            continue  # Продолжаем при ошибках парсинга

    return winks_list

def print_save(txt):
    print(txt)
    with open("results.txt", "a") as f:
        f.write(txt)

async def main():
    """Главная управляющая функция"""
    with open("results.txt", "w") as f:
        f.write("")
    async with ClientSession() as session:  # Создаем сессию
        start_time = time.time()  # Старт замера времени
        # Параллельно запускаем все парсеры
        results = await gather(
            parse_kion(session),
            parse_winks(session),
            computer_science(session)
        )
        # Вывод времени выполнения
        print(f"Общее время выполнения: {time.time() - start_time:.2f} сек.\n\n\n")

    # Формируем итоговый словарь с результатами
    results = {
        "Kion": results[0],
        "Winks": results[1],
        "Computer Science": results[2],
    }


    for name, res in results.items():
        print_save(f"Список ссылок {name}...\n")
        # Вывод результатов
        if not res:
            print_save(f"""❌ Не удалось получить список {name}. Причины:
    - Изменение структуры сайта
    - Проблемы с подключением
    - Блокировка запросов""")
        else:
            print_save(f"✅ Успешно получено {len(res)} записей:\n")
            for i, item in enumerate(res, 1):
                print_save(f"{i}. {item['title']}")
                print_save(f"   🔗 {item['url']}\n")
        print_save("\n\n")

# Запуск в средах с активным event loop (Jupyter/Colab)
if __name__ == "__main__":
    # Обычный запуск для Python скриптов:
    # import asyncio
    # asyncio.run(main())

    # Для Google Colab/Jupyter:
    await main()

Общее время выполнения: 1.41 сек.



Список ссылок Kion...

✅ Успешно получено 10 записей:

1. Тропа гнева
   🔗 https://kion.ru/film/tropa-gneva?_msst=auto&_mbi=0

2. Резервация
   🔗 https://kion.ru/serial/rezervatsiya?_msst=auto&_mbi=0

3. Уволить Жору
   🔗 https://kion.ru/film/uvolit-zhoru?_msst=auto&_mbi=0

4. Отпечатки
   🔗 https://kion.ru/serial/otpechatki?_msst=auto&_mbi=0

5. Равиоли Оли
   🔗 https://kion.ru/film/ravioli-oli?_msst=auto&_mbi=0

6. Папа может
   🔗 https://kion.ru/film/papa-mozhet?_msst=auto&_mbi=0

7. Капкан для монстра
   🔗 https://kion.ru/serial/kapkan-dlya-monstra?_msst=auto&_mbi=0

8. Чебурашка 2
   🔗 https://kion.ru/film/cheburashka-2-2025?_msst=auto&_mbi=0

9. Чёрное солнце
   🔗 https://kion.ru/serial/chernoe-solntse-2024?_msst=auto&_mbi=0

10. Горничная
   🔗 https://kion.ru/film/gornichnaya?_msst=auto&_mbi=0




Список ссылок Winks...

✅ Успешно получено 10 записей:

1. Документальные фильмы о живой природе
   🔗 https://wink.ru/collections/dokumentalnye-fil